In [1]:
import os, json, glob
from collections import defaultdict, OrderedDict
import numpy as np
import pandas as pd
import open3d as o3d
import matplotlib.pyplot as plt
import tqdm as tq


In [ ]:
def downsample_to_ratio(pc, target_ratio, tol=0.02, max_iters=None, accuracy_tol=0.05):
    if target_ratio > 1-1e-3:
        return pc, None
    original_size = len(pc.points)
    target_size = int(original_size * target_ratio)

    aabb = pc.get_axis_aligned_bounding_box()
    extent = aabb.get_extent()
    # tiny lower bound, big upper bound based on scene size
    min_voxel = max(1e-6, float(np.min(extent[extent>0], initial=1e-6)) / 2000.0)
    max_voxel = float(np.max(extent) * 2.0) if np.max(extent) > 0 else 1.0

    best_voxel = None
    if max_iters is not None:
        for _ in range(max_iters):
            mid_voxel = (min_voxel + max_voxel) / 2.0
            pc_down = pc.voxel_down_sample(mid_voxel)
            down_size = len(pc_down.points)

            if abs(down_size - target_size) / target_size < tol:
                best_voxel = mid_voxel
                break

            if down_size > target_size:
                min_voxel = mid_voxel
            else:
                max_voxel = mid_voxel

        if best_voxel is None:
            best_voxel = mid_voxel  # Best effort

        return pc.voxel_down_sample(best_voxel), best_voxel

    else:
        while True:
            mid_voxel = (min_voxel + max_voxel) / 2.0
            pc_down = pc.voxel_down_sample(mid_voxel)
            down_size = len(pc_down.points)

            if abs(down_size - target_size) / target_size < accuracy_tol:
                best_voxel = mid_voxel
                break

            if down_size > target_size:
                min_voxel = mid_voxel
            else:
                max_voxel = mid_voxel

        if best_voxel is None:
            best_voxel = mid_voxel

        return pc.voxel_down_sample(best_voxel), best_voxel

def remove_percent(pc, dir, frac):
    min_bounds = pc.get_min_bound()
    max_bounds = pc.get_max_bound()
    diff = max_bounds - min_bounds
    diff_frac = diff*frac

    curr_points = np.array(pc.points)

    if dir == 'X':
        mask = curr_points[:,0] < diff_frac[0]
    elif dir == 'Y':
        mask = curr_points[:,1] < diff_frac[1]
    elif dir == 'Z':
        mask = curr_points[:,2] < diff_frac[2]
    else:
        raise Exception

    new_points = curr_points[mask]
    new_points = o3d.utility.Vector3dVector(new_points)
    pc.points = new_points
    return pc
    
def remove_volume_fraction(
    pc: o3d.geometry.PointCloud,
    axis: str = "X",
    frac: float = 0.1,
    side: str = "min",
    inplace: bool = False,
) -> o3d.geometry.PointCloud:
    """
    Remove a slab equal to `frac` of the AABB extent along `axis`.
    E.g., axis='X', frac=0.1, side='min' -> removes the lowest 10% of the bbox along X.
    """
    if not (0.0 <= frac <= 1.0):
        raise ValueError("frac must be in [0, 1].")

    # Handle trivial cases up front
    n = len(pc.points)
    if n == 0 or frac == 0.0:
        # Return a shallow copy without using clone()
        out = pc.select_by_index(np.arange(n))
        if inplace:
            pc.points = out.points
            if pc.has_colors() and out.has_colors(): pc.colors = out.colors
            else: pc.colors = o3d.utility.Vector3dVector()
            if pc.has_normals() and out.has_normals(): pc.normals = out.normals
            else: pc.normals = o3d.utility.Vector3dVector()
            return pc
        return out

    if frac == 1.0:
        # Remove everything
        empty = o3d.geometry.PointCloud()
        if inplace:
            pc.points = o3d.utility.Vector3dVector()
            pc.colors = o3d.utility.Vector3dVector()
            pc.normals = o3d.utility.Vector3dVector()
            return pc
        return empty

    ax_idx = {"x": 0, "y": 1, "z": 2}
    ax = axis.lower()
    if ax not in ax_idx:
        raise ValueError("axis must be one of 'X', 'Y', 'Z' (case-insensitive).")
    i = ax_idx[ax]

    minb = pc.get_min_bound()
    maxb = pc.get_max_bound()
    span = maxb[i] - minb[i]

    # Degenerate along this axis: no well-defined slab → just return a copy
    if span <= 0:
        out = pc.select_by_index(np.arange(n))
        if inplace:
            pc.points = out.points
            if pc.has_colors() and out.has_colors(): pc.colors = out.colors
            else: pc.colors = o3d.utility.Vector3dVector()
            if pc.has_normals() and out.has_normals(): pc.normals = out.normals
            else: pc.normals = o3d.utility.Vector3dVector()
            return pc
        return out

    pts = np.asarray(pc.points)
    if side == "min":
        thr = minb[i] + frac * span
        keep_mask = pts[:, i] >= thr
    elif side == "max":
        thr = maxb[i] - frac * span
        keep_mask = pts[:, i] <= thr
    else:
        raise ValueError("side must be 'min' or 'max'.")

    keep_idx = np.flatnonzero(keep_mask)
    out = pc.select_by_index(keep_idx, invert=False)

    if inplace:
        pc.points = out.points
        # Preserve attributes if present
        if pc.has_colors() and out.has_colors(): pc.colors = out.colors
        else: pc.colors = o3d.utility.Vector3dVector()
        if pc.has_normals() and out.has_normals(): pc.normals = out.normals
        else: pc.normals = o3d.utility.Vector3dVector()
        return pc
    else:
        return out

def add_noise(pc, std):
    curr_points = np.array(pc.points)

    curr_points[:] += np.random.normal(0,std,size=curr_points.shape)
    new_points = o3d.utility.Vector3dVector(curr_points)
    pc.points = new_points
    return pc

def add_random_spheres_to_pcd_fraction(pcd, fraction=0.05, radius=0.02, points_per_sphere=200):

    pts = np.asarray(pcd.points)
    if len(pts) == 0:
        raise ValueError("The input point cloud has no points.")

    total_sphere_points = int(len(pts) * fraction)
    num_spheres = max(1, total_sphere_points // points_per_sphere)

    min_bound = pts.min(axis=0)
    max_bound = pts.max(axis=0)

    all_points = [pts]

    for _ in range(num_spheres):
        center = np.random.uniform(min_bound, max_bound)

        phi = np.random.uniform(0, np.pi * 2, points_per_sphere)
        costheta = np.random.uniform(-1, 1, points_per_sphere)
        u = np.random.uniform(0, 1, points_per_sphere)

        theta = np.arccos(costheta)
        r = radius * (u ** (1/3))

        xs = r * np.sin(theta) * np.cos(phi) + center[0]
        ys = r * np.sin(theta) * np.sin(phi) + center[1]
        zs = r * np.cos(theta) + center[2]

        sphere_points = np.vstack((xs, ys, zs)).T
        all_points.append(sphere_points)

    combined_points = np.vstack(all_points)

    pcd_with_spheres = o3d.geometry.PointCloud()
    pcd_with_spheres.points = o3d.utility.Vector3dVector(combined_points)

    return pcd_with_spheres


In [3]:
base_pcd_path = "./subsample_noise_01_20_220_441_gt.pcd"
base_pcd = o3d.io.read_point_cloud(base_pcd_path)

pcd_name = "city_block"
save_path_base = "./"+pcd_name+"_"

In [4]:
def interpolate(start, end, num_steps):
    if num_steps < 2:
        return [start]
    step_size = (end - start) / (num_steps - 1)
    return [start + i * step_size for i in range(num_steps)]

In [5]:
res_degradations = interpolate(0.1, 0.9, 10) 
noise_degradations = interpolate(0.01, 0.1, 10) 
cov_degradations = interpolate(0.1, 0.9, 10)
artifact_degradations = interpolate(0.01, 0.1, 10)

artifact_radius = 0.02
coverage_direction = 'X'

## Resolution Degradation (Downsample)

In [6]:
print ("Running resolution degradation for intensities:", res_degradations)

Running resolution degradation for intensities: [0.1, 0.18888888888888888, 0.2777777777777778, 0.3666666666666667, 0.4555555555555556, 0.5444444444444445, 0.6333333333333333, 0.7222222222222222, 0.8111111111111111, 0.9]


In [7]:
resolution_results = {}
for res in tq.tqdm(res_degradations):
    degraded_pcd, voxel_size = downsample_to_ratio(base_pcd, res, accuracy_tol=0.01)
    resolution_results[res] = {
        "intensity": res,
        "pcd": degraded_pcd,
        "voxel_size": voxel_size,
        "num_points": len(degraded_pcd.points)
    }

100%|██████████| 10/10 [00:27<00:00,  2.77s/it]


In [8]:
base_num_points = len(base_pcd.points)
print ("Resolution Degradation Results:")
for res, data in resolution_results.items():
    recovered_ratio = data['num_points'] / base_num_points if base_num_points > 0 else 0
    accuracy = 1.0 - abs(recovered_ratio - res) / res if res > 0 else 1.0
    print (f"Degradation Intensity: {res:.2f}, Voxel Size: {data['voxel_size']:.4f}, Num Points: {data['num_points']}, Accuracy: {accuracy:.4f}")

Resolution Degradation Results:
Degradation Intensity: 0.10, Voxel Size: 0.1879, Num Points: 129493, Accuracy: 0.9908
Degradation Intensity: 0.19, Voxel Size: 0.1367, Num Points: 243438, Accuracy: 0.9956
Degradation Intensity: 0.28, Voxel Size: 0.1123, Num Points: 354131, Accuracy: 0.9936
Degradation Intensity: 0.37, Voxel Size: 0.0946, Num Points: 468902, Accuracy: 0.9967
Degradation Intensity: 0.46, Voxel Size: 0.0818, Num Points: 581172, Accuracy: 0.9943
Degradation Intensity: 0.54, Voxel Size: 0.0708, Num Points: 695136, Accuracy: 0.9951
Degradation Intensity: 0.63, Voxel Size: 0.0610, Num Points: 805983, Accuracy: 0.9918
Degradation Intensity: 0.72, Voxel Size: 0.0513, Num Points: 917529, Accuracy: 0.9901
Degradation Intensity: 0.81, Voxel Size: 0.0403, Num Points: 1042903, Accuracy: 0.9979
Degradation Intensity: 0.90, Voxel Size: 0.0293, Num Points: 1149376, Accuracy: 0.9953


In [9]:
# Save resolution degraded point clouds
resolution_save_dir = save_path_base + "resolution_degraded/"
os.makedirs(resolution_save_dir, exist_ok=True)
for res, data in resolution_results.items():
    save_path = os.path.join(resolution_save_dir, f"{pcd_name}_res_{res:.2f}_vox_{data['voxel_size']:.4f}.pcd")
    o3d.io.write_point_cloud(save_path, data['pcd'])
    print (f"Saved resolution degraded point cloud at: {save_path}")

Saved resolution degraded point cloud at: ./city_block_resolution_degraded/city_block_res_0.10_vox_0.1879.pcd
Saved resolution degraded point cloud at: ./city_block_resolution_degraded/city_block_res_0.19_vox_0.1367.pcd
Saved resolution degraded point cloud at: ./city_block_resolution_degraded/city_block_res_0.28_vox_0.1123.pcd
Saved resolution degraded point cloud at: ./city_block_resolution_degraded/city_block_res_0.37_vox_0.0946.pcd
Saved resolution degraded point cloud at: ./city_block_resolution_degraded/city_block_res_0.46_vox_0.0818.pcd
Saved resolution degraded point cloud at: ./city_block_resolution_degraded/city_block_res_0.54_vox_0.0708.pcd
Saved resolution degraded point cloud at: ./city_block_resolution_degraded/city_block_res_0.63_vox_0.0610.pcd
Saved resolution degraded point cloud at: ./city_block_resolution_degraded/city_block_res_0.72_vox_0.0513.pcd
Saved resolution degraded point cloud at: ./city_block_resolution_degraded/city_block_res_0.81_vox_0.0403.pcd
Saved reso

## Accuracy Degradation (Noise)

In [10]:
print ("Running accuracy degradation for intensities:", noise_degradations)

Running accuracy degradation for intensities: [0.01, 0.020000000000000004, 0.030000000000000006, 0.04000000000000001, 0.05000000000000001, 0.06000000000000001, 0.07, 0.08, 0.09000000000000001, 0.10000000000000002]


In [11]:
accuracy_results = {}
for noise in tq.tqdm(noise_degradations):
    degraded_pcd = add_noise(base_pcd, noise)
    accuracy_results[noise] = {
        "intensity": noise,
        "pcd": degraded_pcd
    }

100%|██████████| 10/10 [00:00<00:00, 14.42it/s]


In [12]:
# Save degraded point clouds
accuracy_save_dir = save_path_base + "accuracy_degraded/"
os.makedirs(accuracy_save_dir, exist_ok=True)
for acc, data in accuracy_results.items():
    save_path = os.path.join(accuracy_save_dir, f"{pcd_name}_acc_{acc:.4f}.pcd")
    o3d.io.write_point_cloud(save_path, data['pcd'])
    print (f"Saved accuracy degraded point cloud at: {save_path}")

Saved accuracy degraded point cloud at: ./city_block_accuracy_degraded/city_block_acc_0.0100.pcd
Saved accuracy degraded point cloud at: ./city_block_accuracy_degraded/city_block_acc_0.0200.pcd
Saved accuracy degraded point cloud at: ./city_block_accuracy_degraded/city_block_acc_0.0300.pcd
Saved accuracy degraded point cloud at: ./city_block_accuracy_degraded/city_block_acc_0.0400.pcd
Saved accuracy degraded point cloud at: ./city_block_accuracy_degraded/city_block_acc_0.0500.pcd
Saved accuracy degraded point cloud at: ./city_block_accuracy_degraded/city_block_acc_0.0600.pcd
Saved accuracy degraded point cloud at: ./city_block_accuracy_degraded/city_block_acc_0.0700.pcd
Saved accuracy degraded point cloud at: ./city_block_accuracy_degraded/city_block_acc_0.0800.pcd
Saved accuracy degraded point cloud at: ./city_block_accuracy_degraded/city_block_acc_0.0900.pcd
Saved accuracy degraded point cloud at: ./city_block_accuracy_degraded/city_block_acc_0.1000.pcd


## Coverage Degradation (Remove Points)

In [19]:
print ("Running coverage degradation for intensities:", cov_degradations)

Running coverage degradation for intensities: [0.1, 0.18888888888888888, 0.2777777777777778, 0.3666666666666667, 0.4555555555555556, 0.5444444444444445, 0.6333333333333333, 0.7222222222222222, 0.8111111111111111, 0.9]


In [20]:
cov_results = {}
for cov in tq.tqdm(cov_degradations):
    degraded_pcd = remove_volume_fraction(base_pcd, 'X', cov)
    cov_results[cov] = {
        "intensity": cov,
        "pcd": degraded_pcd
    }

100%|██████████| 10/10 [00:01<00:00,  8.33it/s]


In [22]:
base_num_points = len(base_pcd.points)
for cov, data in cov_results.items():
    recovered_ratio = len(data['pcd'].points) / base_num_points if base_num_points > 0 else 0
    print (f"Degradation Intensity: {cov:.2f}, Num Points: {len(data['pcd'].points)}")

Degradation Intensity: 0.10, Num Points: 1178180
Degradation Intensity: 0.19, Num Points: 1085520
Degradation Intensity: 0.28, Num Points: 1022216
Degradation Intensity: 0.37, Num Points: 831103
Degradation Intensity: 0.46, Num Points: 664175
Degradation Intensity: 0.54, Num Points: 480806
Degradation Intensity: 0.63, Num Points: 386896
Degradation Intensity: 0.72, Num Points: 349427
Degradation Intensity: 0.81, Num Points: 296004
Degradation Intensity: 0.90, Num Points: 160092


In [23]:
# Save coverage degraded point clouds
cov_save_dir = save_path_base + "coverage_degraded/"
os.makedirs(cov_save_dir, exist_ok=True)
for cov, data in cov_results.items():
    save_path = os.path.join(cov_save_dir, f"{pcd_name}_cov_{cov:.4f}.pcd")
    o3d.io.write_point_cloud(save_path, data['pcd'])
    print (f"Saved coverage degraded point cloud at: {save_path}")


Saved coverage degraded point cloud at: ./city_block_coverage_degraded/city_block_cov_0.1000.pcd
Saved coverage degraded point cloud at: ./city_block_coverage_degraded/city_block_cov_0.1889.pcd
Saved coverage degraded point cloud at: ./city_block_coverage_degraded/city_block_cov_0.2778.pcd
Saved coverage degraded point cloud at: ./city_block_coverage_degraded/city_block_cov_0.3667.pcd
Saved coverage degraded point cloud at: ./city_block_coverage_degraded/city_block_cov_0.4556.pcd
Saved coverage degraded point cloud at: ./city_block_coverage_degraded/city_block_cov_0.5444.pcd
Saved coverage degraded point cloud at: ./city_block_coverage_degraded/city_block_cov_0.6333.pcd
Saved coverage degraded point cloud at: ./city_block_coverage_degraded/city_block_cov_0.7222.pcd
Saved coverage degraded point cloud at: ./city_block_coverage_degraded/city_block_cov_0.8111.pcd
Saved coverage degraded point cloud at: ./city_block_coverage_degraded/city_block_cov_0.9000.pcd


## Artifact Degradation (Add Outliers)

In [24]:
print ("Running artifact degradation for intensities:", artifact_degradations)

Running artifact degradation for intensities: [0.01, 0.020000000000000004, 0.030000000000000006, 0.04000000000000001, 0.05000000000000001, 0.06000000000000001, 0.07, 0.08, 0.09000000000000001, 0.10000000000000002]


In [25]:
artifact_results = {}
for art in tq.tqdm(artifact_degradations):
    degraded_pcd = add_random_spheres_to_pcd_fraction(base_pcd, fraction=art, radius=artifact_radius)
    artifact_results[art] = {
        "intensity": art,
        "pcd": degraded_pcd
    }

100%|██████████| 10/10 [00:00<00:00, 20.46it/s]


In [27]:
base_num_points = len(base_pcd.points)
for art, data in artifact_results.items():
    recovered_ratio = len(data['pcd'].points) / base_num_points if base_num_points > 0 else 0
    accuracy = 1.0 - abs(recovered_ratio - (1.0 + art)) / (1.0 + art) if (1.0 + art) > 0 else 1.0
    print (f"Degradation Intensity: {art:.2f}, Num Points: {len(data['pcd'].points)}, Accuracy: {accuracy:.4f}")

Degradation Intensity: 0.01, Num Points: 1295898, Accuracy: 1.0000
Degradation Intensity: 0.02, Num Points: 1308698, Accuracy: 1.0000
Degradation Intensity: 0.03, Num Points: 1321498, Accuracy: 0.9999
Degradation Intensity: 0.04, Num Points: 1334298, Accuracy: 0.9999
Degradation Intensity: 0.05, Num Points: 1347098, Accuracy: 0.9999
Degradation Intensity: 0.06, Num Points: 1359898, Accuracy: 0.9999
Degradation Intensity: 0.07, Num Points: 1372898, Accuracy: 1.0000
Degradation Intensity: 0.08, Num Points: 1385698, Accuracy: 1.0000
Degradation Intensity: 0.09, Num Points: 1398498, Accuracy: 0.9999
Degradation Intensity: 0.10, Num Points: 1411298, Accuracy: 0.9999


In [28]:
# Save artifact degraded point clouds
artifact_save_dir = save_path_base + "artifact_degraded/"
os.makedirs(artifact_save_dir, exist_ok=True)
for art, data in artifact_results.items():
    save_path = os.path.join(artifact_save_dir, f"{pcd_name}_art_{art:.4f}.pcd")
    o3d.io.write_point_cloud(save_path, data['pcd'])
    print (f"Saved artifact degraded point cloud at: {save_path}")

Saved artifact degraded point cloud at: ./city_block_artifact_degraded/city_block_art_0.0100.pcd
Saved artifact degraded point cloud at: ./city_block_artifact_degraded/city_block_art_0.0200.pcd
Saved artifact degraded point cloud at: ./city_block_artifact_degraded/city_block_art_0.0300.pcd
Saved artifact degraded point cloud at: ./city_block_artifact_degraded/city_block_art_0.0400.pcd
Saved artifact degraded point cloud at: ./city_block_artifact_degraded/city_block_art_0.0500.pcd
Saved artifact degraded point cloud at: ./city_block_artifact_degraded/city_block_art_0.0600.pcd
Saved artifact degraded point cloud at: ./city_block_artifact_degraded/city_block_art_0.0700.pcd
Saved artifact degraded point cloud at: ./city_block_artifact_degraded/city_block_art_0.0800.pcd
Saved artifact degraded point cloud at: ./city_block_artifact_degraded/city_block_art_0.0900.pcd
Saved artifact degraded point cloud at: ./city_block_artifact_degraded/city_block_art_0.1000.pcd


# Empir3D Evaluation Playground

---

## Test compute

In [34]:
import json
import os
from pathlib import Path
import sys
sys.path.append("/Users/yashturkar/Workspace/3DQualMetrics/Applications/Empir3D_Framework")
from MapMetricManager_v3 import MapMetricManager_v3 as MapMetricManager

# Required inputs
gt_path = "./subsample_noise_01_20_220_441_gt.pcd"        # reference point cloud
cnd_path = "/Users/yashturkar/Workspace/3DQualMetrics/city_block_resolution_degraded/city_block_res_0.54_vox_0.0708.pcd"      # candidate point cloud to be evaluated
cell_size = 5                              # example value, tunable
metric_options = {
    "wc": 0.25, "wt": 0.25, "wa": 0.25, "wr": 0.25,  # weights for coverage, artifact, accuracy, resolution
    "e": 0.1                                       # epsilon (precision threshold)
}

# Output file path
output_file = "output_metrics.json"

# Create and compute metrics
manager = MapMetricManager(gt_path, cnd_path, cell_size, metric_options=metric_options)
manager.compute_metric(output_file)

# View results
with open(output_file, "r") as f:
    results = json.load(f)

# print(json.dumps(results, indent=2))

# Print only average metrics
print("Average Metrics:")
average_metrics = results.get("average", {})
for metric, value in average_metrics.items():
    print(f"{metric}: {value:.4f}") 

Number of Iterations:  99


Creating Jobs: 100%|██████████| 99/99 [00:00<00:00, 191.60it/s]




Number of Jobs:  99
Number of Cores:  9




Running Jobs: 100%|██████████| 11/11 [00:01<00:00,  7.70it/s]


Time Taken:  2.615119695663452
Average  coverage :  0.9239703593579409
Average  artifacts :  0.9963378771710978
Average  resolution :  0.5880703935942534
Average  accuracy :  0.9113969407388738
Average  quality :  0.9113969407388738
Average Metrics:
coverage: 0.9240
artifacts: 0.9963
resolution: 0.5881
accuracy: 0.9114
quality: 0.9114
